# Música em Contexto: Music Brainz + Spotify

## Camada Silver
Aqui eu normalizo dados para fazer o join entre **musicbrainz** e **spotify**, criando a tabela **mb_spotify**. Depois eu faço um novo join entre **mb_spotify** e a tabela **mb_artist** usando **id** e **artist_mbids** para formar a tabela **contexto_silver**.

No processo de normalização e join a base de dados passou de ??????? instâncias (na tabela **musicbrainz** originária) para ???????? instâncias na tabela **mb_spotify** e ??????? na tabela **contexto_silver**.

COLOCAR AQUI OS DETALHES DIREITO

In [0]:
%pip install unidecode

In [0]:
%restart_python

In [0]:
from unidecode import unidecode
import re

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [0]:
SEED = 42

In [0]:
%sql
USE CATALOG mvp;
USE SCHEMA silver;

Definindo **musicbrainz** como **df_musicbrainz**

In [0]:
df_musicbrainz = spark.read.table("mvp.bronze.musicbrainz")

display(df_musicbrainz.limit(10))

Selecionando em **df_musicbrainz** apenas as colunas que interessam ao trabalho:
- artist_mbids: ids dos artistas, chave para fazer o join com **mb_artist**
- artist_credit_name
- release_name
- recording_name
- combined_lookup: uma chave que eu vou ter que reproduzir em **spotify** para fazer o join das duas tabelas

In [0]:
df_musicbrainz = df_musicbrainz.select("artist_mbids", "artist_credit_name", "release_name", "recording_name", "combined_lookup")

display(df_musicbrainz.limit(10))

### Testando Reproduzir o Combined_Lookup
O atributo **combined_lookup** é a chave para fazer o join entre **musicbrainz** e **spotify**.

Vou testar a minha capacidade de reproduzi-lo com os próprios dados da **df_musicbrainz**, uma vez tendo sucesso, vou reproduzir essas transformações criando uma coluna chave em **df_spotify** para fazer através dela o join entre as duas tabelas.

O **combined_lookup** foi produzido unindo **artist_credit_name** e **release_name**, depois transformando os caracteres não-ASCII em ASCII e retirando espaços e caixas altas.

Examinando o **combined_lookup** eu percebo que ele contém erros: espaços e caixas altas que resistiram à normalização de MusicBrainz. É provavel que o meu método resolva esses erros.

In [0]:
from pyspark.sql import functions as F

In [0]:
contagem_esp_maiusc = (
    df_musicbrainz
    .select(
        F.count("*").alias("total"),
        F.sum(
            F.when(F.col("combined_lookup").contains(" "), 1).otherwise(0)
        ).alias("com_espacos"),
        F.sum(
            F.when(F.col("combined_lookup") != F.lower(F.col("combined_lookup")), 1).otherwise(0)
        ).alias("com_maiusculas"),
        F.sum(
            F.when(F.col("combined_lookup").contains(" ") |
                   (F.col("combined_lookup") != F.lower(F.col("combined_lookup"))), 1).otherwise(0)
                   ).alias("espacos_eou_maiusculas")
        )
        )
contagem_esp_maiusc = contagem_esp_maiusc.withColumn("porcentagem_do_total", F.lit(
    F.round(F.col("espacos_eou_maiusculas")*100/F.col("total"), 2)
    ))

display(contagem_esp_maiusc)

Existem aproximadamente 5% de linhas com erros em **combined_lookup** na tabela **df_musicbrainz**. Essa é a chave para conectar com a **df_spotify**. Será que eu consigo diminuir um pouco a quantidade de erros?

Primeira etapa normalização para **combined_lookup**: transformar non-ASCII em ASCII criando a coluna **dummy_cl** em **df_musicbrainz**

Para isso tive que instalar **unidecode** e **udf**

Criando a primeira função de normalização:

In [0]:
def faz_dummy_cl_1(artist_name, recording_name):
    return unidecode(
        re.sub(
            r'[^\w]+',
            '',
            artist_name + recording_name
        ).lower()
    )

In [0]:
faz_dummy_cl = udf(
    faz_dummy_cl_1,
    StringType()
)

In [0]:
df_musicbrainz = df_musicbrainz.withColumn(
    "dummy_cl",
    faz_dummy_cl(
        F.col("artist_credit_name"),
        F.col("recording_name")
    )
)

display(
    df_musicbrainz
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200))

O meu **dummy_cl** está bem parecido com o **combined_lookup** original. Agora vou aplicar uma segunda normalização apenas no **dummy_cl**.

In [0]:
def faz_normal(df, coluna_original, coluna_normalizada):
    return df.withColumn(
        coluna_normalizada,
        F.lower(
            F.regexp_replace(
                F.col(coluna_original),
                r"\s+",
                ""
            )
        )
    )

In [0]:
df_musicbrainz = faz_normal(df_musicbrainz, "dummy_cl", "cl_normal")

display(
    df_musicbrainz
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200)
    .drop("artist_credit_id", "artist_mbids", "release_name", "release_mbid", "recording_mbid", "score")
    )

O resultado da chave feita a partir das duas funções de normalização criadas nesse projeto é superior: tem menos espaços e caixas altas. Vou usar a cl_normal como chave para fazer o join entre **df_musicbrainz** e **df_spotify**

Então vou cortar da **df_musicbrainz** **combined_lookup** e **dummy_cl**

In [0]:
df_musicbrainz = df_musicbrainz.drop("combined_lookup", "dummy_cl")

### Aplicando a Normalização em Spotify
Aplicando as duas funções de normalização **faz_dummy_cl** e **faz_normal** para criar uma coluna que sirva como chave para o join entre **musicbrainz** e **spotify**

In [0]:
df_spotify = spark.read.table("mvp.bronze.spotify")

display(df_spotify.limit(10))

Retirando algumas colunas desencessárias para o projeto:

In [0]:
df_spotify = df_spotify.drop("track_id", "track_number", "disc_number", "album_popularity", "track_vs_album_popularity", "album_vs_artist_popularity")

display(df_spotify.limit(10))

Criando **dummy_cl_spotify** com a função **faz_dummy_cl** que une **artist_name** e **track_name** e transforma caracteres não-ASCII

In [0]:
df_spotify = df_spotify.withColumn(
    "dummy_cl_spotify",
    faz_dummy_cl(
        F.col("artist_name"),
        F.col("track_name")
    )
)

display(
    df_spotify
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200))

Agora aplicando a segunda normalização com a função **faz_normal**

In [0]:
df_spotify = faz_normal(df_spotify, "dummy_cl_spotify", "cl_normal_spotify")

display(
    df_musicbrainz
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200)
    )

Retirando **dummy_cl_spotify** da **df_spotify**

In [0]:
df_spotify = df_spotify.drop("dummy_cl_spotify")

### Salvando as Tabelas com as Chaves

Aqui os dataframes **df_musicbrainz** e **df_spotify** são salvos como tabelas **musicbrainz_chave** e **spotify_chave** na camada **Silver**

In [0]:
df_musicbrainz.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp.silver.musicbrainz_chave")

In [0]:
df_spotify.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp.silver.spotify_chave")

### Fazendo Join Entre df_musicbrainz e df_spotify
Agora é feito o primeiro join entre as dataframes **df_musicbrainz** e **df_spotify**, usando as chaves criadas nesse estudo: **cl_normal** em **df_musicbrainz** e **cl_normal_spotify** em **df_spotify** e criando a nova dataframe **df_mb_spotify**

In [0]:
df_mb_spotify = (
    df_musicbrainz
    .join(
        df_spotify,
        df_musicbrainz.cl_normal == df_spotify.cl_normal_spotify,
        how="inner"
    )
)

display(
    df_mb_spotify
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200)
    .drop()
    )

Salvando **df_mb_spotify** como a tabela **mb_spotify** na camada **Silver**

In [0]:
df_mb_spotify.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp.silver.mb_spotify")

### Fazendo Join Entre df_mb_spotify e df_mb_artist
Essas duas dataframes vão ser ligadas através da chave **id** do MusicBrainz. Em **df_mb_spotify** esse **id** aparece como **artist_mbids** e em **df_mb_artist** aparece simplesmente como **id**

In [0]:
df_mb_artist = spark.read.table("mvp.bronze.mb_artist")

display(df_mb_artist.limit(10))

Fazendo o join para criar **df_contexto_silver**

In [0]:
df_contexto_silver = (
    df_mb_spotify
    .join(
        df_mb_artist,
        df_mb_spotify.artist_mbids == df_mb_artist.id,
        how="inner"
    )
)

display(
    df_contexto_silver
    .sample(withReplacement=False, fraction=0.03, seed=SEED)
    .limit(200)
    .drop()
    )

Existem valores nulos em **country** que serão tratados na camada **Gold**.

Agora vou salvar **df_contexto_silver** como a tabela **contexto_silver** na camada **silver**

In [0]:
df_contexto_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp.silver.contexto_silver")